# 01. Download, extract, and inspect walking sequences

We will trace each skeleton back to a current CSV, decode the annotated person, check frame timing, and produce reproducible pose caches. This notebook can preview ingestion independently of model training.

## Run this notebook independently

From `experiments/aaai`, run:

```bash
uv sync
uv run jupyter lab
```

Select the kernel from this environment. Python 3.12 and all notebook libraries are declared in `pyproject.toml`. Every notebook repeats its setup and loads its own inputs. No variable from another notebook is required.

A fresh installation defaults to `AAAI_DEMO=1` for a small synthetic software check. This prepared workspace uses `AAAI_DEMO=0` after real-data ingestion and validation. Its condition parameters are invented and its scores are not research evidence. Set `AAAI_DEMO=0` in the repository root `.env` after the ingestion workflow has produced a complete pose corpus. All notebooks use the same root `.env`. No API key is needed for JEPA or Random Forest training. Restart the kernel after changing configuration.

Only the root environment file contains credentials. `load_env` and `find_env` below are aliases for the real `python-dotenv` functions, `load_dotenv` and `find_dotenv`.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys
# Find the repository from either its root or this notebook directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ambient').is_dir() and (p / 'data/gavd').is_dir())
BASE = ROOT / 'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())  # requested spelling, using aliases for python-dotenv's real API
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.settings import Settings
from aaai_gait.data import build_manifest, grouped_split, LABELS
from aaai_gait.media import show_video_context, plot_motion
settings = Settings()
torch.set_num_threads(2)
manifest = grouped_split(build_manifest(ROOT), seed=settings.seed)
print('Mode:', 'SYNTHETIC SOFTWARE DEMONSTRATION' if settings.demo else 'GAVD RESEARCH RUN')
print('Current annotation sequences:', len(manifest), 'unique sources:', manifest.source_id.nunique())

## Connect the experiment to walking

Watch a short annotated normal training sequence when local media is available. The preview is an observed video. Synthetic arrays used below are a separate software demonstration.

In [ ]:
video_row = show_video_context(manifest, frame_number_base=settings.frame_number_base)

## 1. Keep one source video in one place

Every new download goes to `data/videos-gavd/<source_id>/<source_id>.mp4` or the source container's extension. Existing valid flat files in `data/videos-gavd` are reused. A clip with two conditions is still one source. We do not duplicate it into class folders.

Download failures are records, not permission to replace the dataset with whatever happens to be cached. The CSV manifest is rebuilt from the five selected folders each time.

![Ingestion](images/data_pipeline.svg)

In [ ]:
from aaai_gait.data import download_sources, find_video
# A preview is deliberately small. Use scripts/run_pipeline.py ingest for the full cohort.
if not settings.demo:
    limit = int(os.getenv('AAAI_DOWNLOAD_LIMIT', '3'))
    preview_manifest=manifest[manifest.label.eq('normal') & manifest.split.eq('train')]
    preview_sources=preview_manifest.drop_duplicates('source_id').sort_values('source_id').head(limit)
    if all(find_video(ROOT,source) is not None for source in preview_sources.source_id):
        print('The preview training sources are already locally decodable.')
    else:
        download_report = download_sources(preview_manifest, ROOT, limit=limit)
        display(download_report[['source_id','status','error']])
    video_row=show_video_context(manifest,frame_number_base=settings.frame_number_base)
else:
    print('Demo mode: no downloads. Existing local videos may still be previewed.')

## 2. Verify the annotated frame and person

GAVD stores source frame numbers, an image size, and a person bounding box. The repository adapter treats frame numbers as one-based. OpenCV uses zero-based decoded positions, so frame 1 maps to decoded index 0. The configurable frame base is recorded in the pose cache. Verify several actual overlays before a large run.

Bounding boxes must scale when a downloaded video has a different resolution. We crop the annotated box with a small margin and restore detected points to image coordinates. A crop helps associate the pose with the annotated person, but it cannot guarantee identity when people overlap.

In [ ]:
from aaai_gait.media import annotated_frame
if video_row is not None:
    annotated_frame(video_row, frame_number_base=settings.frame_number_base)
    plt.show()
else:
    print('An overlay becomes available after the first successful download.')

## 3. Extract with ambient

`SequenceKeypointExtractor` supplies the MediaPipe model and extracts each annotated crop. Raw coordinates, confidence, source FPS, decoded-frame flags, and source frame indices are retained. Cache provenance includes the CSV, video, pose-model hash, visibility threshold, and extraction settings. A stale cache raises an error instead of silently reusing it.

All 33 raw points can be stored for inspection. The learning pipeline selects only the declared 16-joint whitelist. Confidence defines observation validity and is not supplied as a numerical encoder feature. Missingness can still be a shortcut, so the capstone includes a pose-availability control.

In [ ]:
from aaai_gait.data import extract_sequence, load_pose
if not settings.demo and video_row is not None:
    pose_path=extract_sequence(video_row, find_video(ROOT,video_row['source_id']), settings.pose_dir,
                               frame_number_base=settings.frame_number_base)
    pose=load_pose(pose_path)
    print('Frames:',len(pose['frame_numbers']),'FPS:',float(pose['fps']))
    print('Decoded fraction:',pose['decoded'].mean())
    print('Selected-joint visibility fraction:',(pose['confidence'][:,[11,12,13,14,15,16,23,24,25,26,27,28,29,30,31,32]]>=.5).mean())
else:
    print('Extraction is enabled by AAAI_DEMO=0. No cached MS dataset is read.')

## 4. Preserve time and audit exclusions

The common corpus uses 15 Hz, 32-frame windows, and a 16-frame stride. Interpolation only joins adjacent observed source frames. Sources below the requested sample rate are excluded instead of upsampled. It never bridges a missing annotation gap. Each four-frame joint token is valid only when its coordinates are observed throughout the patch. A retained window needs at least 75% valid tokens and at least 50% valid tokens in each half. Missing values remain NaN and cannot become teacher targets. This gate can still reject substantial data. Report attrition by class and source, including partially rejected sequences.

Each frame is centered on its pelvis. Scale comes from median torso length in the first half of the window, so future torso measurements cannot change the observed prefix. This is relative image-plane motion, not physical walking speed or depth.

To ingest all selected videos locally:

```bash
uv run python scripts/run_pipeline.py ingest
uv run python scripts/run_pipeline.py audit
```

The first command can take hours and encounters unavailable media. The second reports retained data and refuses to start a five-class evaluation if any split has lost a class. Do not change the split after inspecting performance.

In [ ]:
from aaai_gait.data import load_windows
preview_x,preview_meta=load_windows(manifest,settings.pose_dir,frames=settings.frames,stride=settings.stride,target_fps=settings.target_fps,require_coverage=False,
                                   expected_frame_number_base=settings.frame_number_base)
print('Locally available valid windows:',len(preview_x))
display(pd.DataFrame(preview_meta.attrs['exclusions']).groupby(['label','reason']).size().rename('sequences').reset_index() if preview_meta.attrs['exclusions'] else pd.DataFrame())

## What to record before a paper run

Save the current annotation hash, source assignments, retained windows, exclusions, software versions, model configuration, seed, and exact command. Keep synthetic artifacts separate. Revisit the video when an apparent anomaly could instead be a crop error, occlusion, or viewpoint change.

The [research review](docs/literature_review.md) explains source methods and adaptations. The [paper plan](docs/paper_plan.md) defines claims, ablations, and limits.